## Setup Colab and sync


In [1]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os


Cloning into 'bert_for_patents'...
remote: Enumerating objects: 2369, done.
remote: Counting objects: 100% (627/627), done.
remote: Compressing objects: 100% (292/292), done.
remote: Total 2369 (delta 372), reused 544 (delta 302), pack-reused 1742 (from 2)
Receiving objects: 100% (2369/2369), 210.73 MiB | 10.66 MiB/s, done.
Resolving deltas: 100% (1331/1331), done.


In [2]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
import os
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
Already up to date.


In [3]:
#@title Run setup file to get environment
!python "/content/bert_for_patents/05 Analysis/01 Main/__ aux/setup_colab.py"

absl-py==2.0.0 is already installed.
Installing asttokens==2.4.1...
astunparse==1.6.3 is already installed.
attrs==26.1.0 is already installed.
beautifulsoup4==4.13.5 is already installed.
Installing bert==2.2.0...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for bert: filename=bert-2.2.0-py3-none-any.whl size=3816 sha256=5d0c317d035912670e6d292bc7f6e4a7845b38c0916fc51234040c3221463f00
  Stored in directory: /root/.cache/pip/wheels/cd/64/ff/6d3c6dba947a506f45d8ff1e5b60c219c3eba2949cb77fa4dc
  Created wheel for erlastic: filename=erlastic-2.0.0-py3-none-any.whl size=6839 sha256=11f550e8dd40d55462e2fcdb2148390ee3d68ed3733cf6a13c5944689d7f1751
  Stored in directory: /root/.cache/pip/wheels/10/bd/c6/7ad6665fdaa97581ae44881e02429c28e9e4cc80353eeed277
Suc

In [4]:
!ls

'00 Make patents data rectangular.ipynb'  '04 Colab'	 __init__.py
'00 Python data'			   api		 logdir
'01 Stata'				  '__ aux'	 path_utils.py
'03 Python scripts'			   config.json


In [5]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


Set precision

## Load the patents and process

In [6]:

import os
import sys
import json
import time
import numpy as np
import pandas as pd
import dill as pickle
import tensorflow as tf

from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset


# Check GPU
print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("No GPU detected by TensorFlow!")

import tensorflow as tf

tf.config.experimental.enable_tensor_float_32_execution(True)
tf.config.optimizer.set_jit(True)

import api.bertembeddings as be
import importlib
importlib.reload(be)

EMBEDDING_DIM = 1024




class PatentDataset(Dataset):
    def __init__(self, df):
        self.abstracts = df["abstract"].tolist()

    def __len__(self):
        return len(self.abstracts)

    def __getitem__(self, idx):
        return idx, self.abstracts[idx]


def collate_fn(batch):
    rows, abstracts = zip(*batch)

    embeddings = be.get_embd_of_whole_abstract(
        list(abstracts),
        return_cls_embedding=True,
        has_context_token=True
    )

    embeddings = np.asarray(
        embeddings, dtype=np.float16
    ).reshape(-1, EMBEDDING_DIM)

    return np.asarray(rows, dtype=np.int64), embeddings


def load(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


TensorFlow: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
BERT loaded...
BERT loaded...


In [7]:

# Load applications
path = "/content/drive/MyDrive/uc3m PhD/07 Continuous innovation space/02 Analysis/01 Main/01 Data load/out/02 All applications with an abstract - app_id as index.parquet"
df = pd.read_parquet(path, columns=['pgpub_id', 'patent_id', 'consolidated_abstract'])
df.rename(columns = {'consolidated_abstract':'abstract'}, inplace=True)
df["abstract_source"] = np.where(
    df["pgpub_id"].notna(),
    "pre-grant publication",
    "grant patent"
)


In [8]:
# Validate data
assert df.index.is_unique
assert df.index.notna().all()
assert df["abstract"].notna().all()
# Standardize ID types
df.index = df.index.astype("string")
df.index.name = "application_id"
df["patent_id"] = df["patent_id"].astype("string")

# Create the row-to-application mapping
index_df = df[["patent_id", "abstract_source"]].copy()

index_df.insert(0, "application_id", df.index.to_numpy())
index_df = index_df.reset_index(drop=True)

index_df.insert(
    0, "embedding_row", np.arange(len(df), dtype=np.int64)
)


In [9]:
# Output files
SAVE_FOLDER = Path(
    "/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/Applications"
)
CHUNK_FOLDER = SAVE_FOLDER / "chunks"
CHUNK_FOLDER.mkdir(parents=True, exist_ok=True)

INDEX_FILE = SAVE_FOLDER / "embedding_index.parquet"
MODEL_INFO_FILE = SAVE_FOLDER / "model_info.json"

BATCH_SIZE = 256
SAVE_INTERVAL = 60 * 60  # Every 60 minutes


# Model metadata
model_info = {
    "model_dir": str(be.MODEL_DIR),
    "vocab": str(be.VOCAB),
    "max_seq_length": be.MAX_SEQ_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "context_token": "abstract",
    "storage_dtype": "float16",
    "tensorflow_version": tf.__version__,
}


# Save metadata
if INDEX_FILE.exists():
    if not pd.read_parquet(INDEX_FILE).equals(index_df):
        raise ValueError("Input data or row order changed!")
else:
    index_df.to_parquet(INDEX_FILE, index=False)

if MODEL_INFO_FILE.exists():
    if json.loads(MODEL_INFO_FILE.read_text()) != model_info:
        raise ValueError("Model configuration changed!")
else:
    MODEL_INFO_FILE.write_text(json.dumps(model_info, indent=2))



In [24]:
processed_count

5376

In [10]:
# Resume only chunks from THIS new run
processed_count = 0

for file in sorted(CHUNK_FOLDER.glob("embeddings_*.npy")):
    _, start, end = file.stem.split("_")
    start, end = int(start), int(end)

    if start != processed_count:
        raise ValueError(f"Missing or overlapping chunk: {file}")

    chunk = np.load(file, mmap_mode="r")
    if chunk.shape != (end - start, EMBEDDING_DIM):
        raise ValueError(f"Invalid chunk: {file}")

    processed_count = end
    del chunk

saved_count = processed_count

# Embeddings accumulate in CPU RAM
ram_batches = []


def save_checkpoint():
    global saved_count

    if not ram_batches:
        return

    end = processed_count
    batch = np.concatenate(ram_batches, axis=0)

    if batch.shape != (end - saved_count, EMBEDDING_DIM):
        raise ValueError("Checkpoint shape mismatch!")

    filename = (
        CHUNK_FOLDER /
        f"embeddings_{saved_count:09d}_{end:09d}.npy"
    )

    temp = filename.with_suffix(".tmp")

    with open(temp, "wb") as f:
        np.save(f, batch)

    os.replace(temp, filename)

    saved_count = end
    ram_batches.clear()
    del batch

    print(f"Checkpoint saved: {saved_count:,}/{len(df):,}")


print(f"Total applications: {len(df):,}")
print(f"Starting from row: {processed_count:,}")


# DataLoader
dataset = PatentDataset(df)

dataloader = DataLoader(
    Subset(dataset, range(processed_count, len(dataset))),
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
    shuffle=False,
    num_workers=0
)



Total applications: 11,469,409
Starting from row: 8,001,024


# TEST & COMPARISON RUN

In [8]:
# TEST RUN
import dill as pickle
# Load old embedding dictionary
old_path = "/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/all_embeddings_float16.pkl"

# Load old embeddings
with open(old_path, "rb") as f:
    old_embeddings = pickle.load(f)


In [9]:

# Select 1000 patents available in both datasets
test_df = df[
    df["patent_id"].astype(str).isin(old_embeddings)
].sample(n=2000, random_state=42)

# Get old embeddings
old_array = np.stack([
    np.asarray(old_embeddings[pid]).reshape(1024)
    for pid in test_df["patent_id"]
])


In [12]:
import gc 
import itertools
old_embeddings = dict(itertools.islice(old_embeddings.items(), 10000))

In [14]:
len(old_embeddings)

10000

In [13]:
gc.collect()

15434

In [ ]:
with open("/content/drive/MyDrive/uc3m PhD/PhD Data/01 CLS Embeddings/All embeddings - float16/test_embeddings_float16.pkl", "wb") as f:
    pickle.dump(old_embeddings, f)


In [16]:
# Calculate new embeddings
new_batches = []

batch_size = 128

for i in range(0, len(test_df), batch_size):
    abstracts = test_df["abstract"].iloc[i:i + batch_size].tolist()

    new_batches.append(
        be.get_embd_of_whole_abstract(
            abstracts,
            return_cls_embedding=True,
            has_context_token=True
        )
    )

new_array = np.vstack(new_batches)

assert len(new_array) == len(test_df)

# Compare cosine similarity
from sklearn.metrics.pairwise import paired_cosine_distances

test_df["cosine_similarity"] = 1 - paired_cosine_distances(
    old_array.astype(np.float16),
    new_array.astype(np.float16)
)

print(test_df["cosine_similarity"].describe())

There are 128 patents being fed into BERT
It took 0.06149 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06153 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06140 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06156 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06151 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06154 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06165 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06162 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06153 seconds to get the embeddings of the input.
There are 128 patents being fed into BERT
It took 0.06161 seconds to get the embed

In [23]:
display(test_df.cosine_similarity.describe(percentiles=[0.01, 0.05]))

print(len(test_df[test_df.cosine_similarity < 0.9]) / len(test_df))

,cosine_similarity
count,2000.000000
mean,0.995163
std,0.025801
min,0.617446
1%,0.859265
5%,0.977116
50%,0.999999
max,1.000000


0.0155


In [18]:
test_df[["patent_id", "cosine_similarity"]].sort_values(
    "cosine_similarity"
).head(200)

,patent_id,cosine_similarity
application_id,,
11497926,7480593,0.617446
15380921,11447861,0.621531
16558597,11660590,0.700497
15550293,10218575,0.732825
14907396,9656995,0.769087
...,...,...
12612395,8260951,0.999998
09838981,7595914,0.999998
12270653,8332341,0.999998


# MAIN LOOP

In [11]:

# Main loop — embeddings accumulate in CPU RAM

start_time = time.time()
last_progress_time = start_time
last_save_time = start_time
processed_in_last_minute = 0

try:
    for batch_rows, batch_embeddings in dataloader:

        batch_size = len(batch_rows)
        end = processed_count + batch_size

        # Verify row order
        expected = np.arange(processed_count, end)

        if not np.array_equal(batch_rows, expected):
            raise ValueError("Unexpected row ordering!")

        # Store embeddings in CPU RAM only
        ram_batches.append(
            np.asarray(batch_embeddings, dtype=np.float16).copy()
        )

        processed_count = end
        processed_in_last_minute += batch_size

        current_time = time.time()

        # Progress every 30 seconds
        if current_time - last_progress_time >= 120:

            elapsed = current_time - last_progress_time
            speed = processed_in_last_minute / elapsed

            print(
                f"Last {elapsed:.0f} sec: "
                f"{processed_in_last_minute:,} | "
                f"Total: {processed_count:,}/{len(df):,} | "
                f"Speed: {speed:.1f} abstracts/sec | "
                f"Unsaved: {processed_count-saved_count:,}"
            )

            last_progress_time = current_time
            processed_in_last_minute = 0

        # Save only once every 60 minutes
        if current_time - last_save_time >= SAVE_INTERVAL:
            save_checkpoint()
            last_save_time = time.time()

finally:
    # Save remaining RAM buffer on interruption
    print("Saving remaining embeddings...")
    save_checkpoint()

print("Embedding generation completed!")


Last 121 sec: 9,984 | Total: 8,011,008/11,469,409 | Speed: 82.5 abstracts/sec | Unsaved: 9,984
Last 120 sec: 16,896 | Total: 8,027,904/11,469,409 | Speed: 140.5 abstracts/sec | Unsaved: 26,880
Last 121 sec: 16,896 | Total: 8,044,800/11,469,409 | Speed: 140.2 abstracts/sec | Unsaved: 43,776
Last 121 sec: 16,896 | Total: 8,061,696/11,469,409 | Speed: 140.1 abstracts/sec | Unsaved: 60,672
Last 120 sec: 16,896 | Total: 8,078,592/11,469,409 | Speed: 140.3 abstracts/sec | Unsaved: 77,568
Last 121 sec: 16,896 | Total: 8,095,488/11,469,409 | Speed: 140.1 abstracts/sec | Unsaved: 94,464
Last 120 sec: 16,896 | Total: 8,112,384/11,469,409 | Speed: 140.4 abstracts/sec | Unsaved: 111,360
Last 120 sec: 16,896 | Total: 8,129,280/11,469,409 | Speed: 140.3 abstracts/sec | Unsaved: 128,256
Last 121 sec: 16,896 | Total: 8,146,176/11,469,409 | Speed: 140.0 abstracts/sec | Unsaved: 145,152
Last 121 sec: 16,896 | Total: 8,163,072/11,469,409 | Speed: 140.1 abstracts/sec | Unsaved: 162,048
Last 121 sec: 16,89

In [ ]:
#@title Main loop

try:
    start_time = time.time()
    last_progress_time = start_time
    last_save_time = start_time
    processed_in_last_minute = 0
    for batch_rows, batch_embeddings in dataloader:

        batch_size = len(batch_rows)
        end = processed_count + batch_size

        # Verify correct row ordering
        expected = np.arange(processed_count, end)
        if not np.array_equal(batch_rows, expected):
            raise ValueError("Unexpected row ordering!")

        # Save embeddings directly into NumPy array
        embeddings[batch_rows] = batch_embeddings

        processed_count = end
        processed_in_last_minute += batch_size

        current_time = time.time()

        # Progress every minute
        if current_time - last_progress_time >= 30:

            elapsed = current_time - last_progress_time

            speed = processed_in_last_minute / elapsed

            print(
                f"Last 30 sec: {processed_in_last_minute:,} | "
                f"Total: {processed_count:,}/{len(df):,} | "
                f"Speed: {speed:.1f} abstracts/sec"
            )

            last_progress_time = current_time
            processed_in_last_minute = 0

        # Checkpoint every 90 minutes
        if current_time - last_save_time >= SAVE_INTERVAL:
            save_checkpoint(processed_count)
            last_save_time = current_time

finally:
    # Save even if interrupted or an error occurs
    print("Saving a checkpoint because of KeyboardInterrupt")
    save_checkpoint(processed_count)

print("Embedding generation completed!")


Last 30 sec: 4,352 | Total: 4,352/11,469,409 | Speed: 139.0 abstracts/sec
Last 30 sec: 4,352 | Total: 8,704/11,469,409 | Speed: 139.3 abstracts/sec
Last 30 sec: 4,352 | Total: 13,056/11,469,409 | Speed: 139.2 abstracts/sec
Last 30 sec: 4,352 | Total: 17,408/11,469,409 | Speed: 138.9 abstracts/sec
Last 30 sec: 4,352 | Total: 21,760/11,469,409 | Speed: 138.6 abstracts/sec
Last 30 sec: 4,352 | Total: 26,112/11,469,409 | Speed: 137.7 abstracts/sec
Last 30 sec: 1,536 | Total: 27,648/11,469,409 | Speed: 7.8 abstracts/sec
Last 30 sec: 4,352 | Total: 32,000/11,469,409 | Speed: 139.3 abstracts/sec
Last 30 sec: 4,352 | Total: 36,352/11,469,409 | Speed: 139.5 abstracts/sec
Last 30 sec: 4,352 | Total: 40,704/11,469,409 | Speed: 138.9 abstracts/sec
Last 30 sec: 4,352 | Total: 45,056/11,469,409 | Speed: 138.3 abstracts/sec
Last 30 sec: 4,352 | Total: 49,408/11,469,409 | Speed: 139.4 abstracts/sec
Last 30 sec: 4,352 | Total: 53,760/11,469,409 | Speed: 139.2 abstracts/sec
Last 30 sec: 4,352 | Total: 5

: 

In [19]:
import time

iterator = iter(dataloader)

for i in range(5):
    t0 = time.perf_counter()

    rows, batch = next(iterator)

    t1 = time.perf_counter()

    embeddings[rows[0]:rows[0] + len(rows)] = batch

    t2 = time.perf_counter()

    print(
        f"BERT: {t1-t0:.2f}s | "
        f"Drive write: {t2-t1:.2f}s | "
        f"Speed: {len(rows)/(t2-t0):.1f}/sec"
    )

There are 128 patents being fed into BERT
It took 0.01322 seconds to get the embeddings of the input.
BERT: 1.69s | Drive write: 0.00s | Speed: 75.6/sec
There are 128 patents being fed into BERT
It took 0.01303 seconds to get the embeddings of the input.
BERT: 1.67s | Drive write: 0.00s | Speed: 76.6/sec
There are 128 patents being fed into BERT
It took 0.01304 seconds to get the embeddings of the input.
BERT: 1.67s | Drive write: 0.00s | Speed: 76.6/sec
There are 128 patents being fed into BERT
It took 0.01306 seconds to get the embeddings of the input.
BERT: 1.67s | Drive write: 0.00s | Speed: 76.4/sec
There are 128 patents being fed into BERT
It took 0.01295 seconds to get the embeddings of the input.
BERT: 1.66s | Drive write: 0.00s | Speed: 77.1/sec


In [22]:
import time

texts = df["abstract"].iloc[:512].tolist()

for bs in [32, 64, 128, 256, 512]:
    try:
        # Warm-up
        be.get_embd_of_whole_abstract(texts[:bs])

        start = time.perf_counter()

        for i in range(0, len(texts), bs):
            be.get_embd_of_whole_abstract(texts[i:i+bs])

        elapsed = time.perf_counter() - start

        print(f"Batch {bs}: {len(texts)/elapsed:.1f} abstracts/sec")

    except tf.errors.ResourceExhaustedError:
        print(f"Batch {bs}: GPU out of memory")
        break

There are 32 patents being fed into BERT
It took 0.01431 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01341 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01350 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01344 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01342 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01326 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01327 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01331 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01340 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.01338 seconds to get the embeddings of t

In [24]:
import time
import numpy as np
import tensorflow as tf

p = be.bert_predictor
texts = df["abstract"].iloc[:256].tolist()

for i in range(3):
    t0 = time.perf_counter()

    # CPU tokenization, truncation and padding
    inputs = p.get_features_from_texts(texts)
    inputs = p.add_input_context(
        inputs, ["abstract"] * len(texts)
    )

    t1 = time.perf_counter()

    # Prepare TensorFlow tensors
    kwargs = {
        name: tf.convert_to_tensor(inputs[name], dtype=tf.int64)
        for name in ["input_ids", "input_mask", "segment_ids"]
    }

    kwargs["mlm_positions"] = tf.zeros(
        (len(texts), p.max_preds_per_seq), dtype=tf.int64
    )

    t2 = time.perf_counter()

    # Actual TensorFlow inference
    response = p.model(**kwargs)
    cls = response["cls_token"].numpy()

    t3 = time.perf_counter()

    print(
        f"Tokenization: {t1-t0:.2f}s | "
        f"Tensor conversion: {t2-t1:.2f}s | "
        f"Model execution: {t3-t2:.2f}s | "
        f"Throughput: {len(texts)/(t3-t0):.1f}/sec"
    )

Tokenization: 0.34s | Tensor conversion: 0.09s | Model execution: 2.96s | Throughput: 75.4/sec
Tokenization: 0.35s | Tensor conversion: 0.08s | Model execution: 2.91s | Throughput: 76.6/sec
Tokenization: 0.35s | Tensor conversion: 0.08s | Model execution: 2.90s | Throughput: 76.8/sec


In [25]:
print(
    "TF32 enabled:",
    tf.config.experimental.tensor_float_32_execution_enabled()
)

TF32 enabled: True


In [26]:
import time
import numpy as np
import tensorflow as tf

p = be.bert_predictor
texts = df["abstract"].iloc[:256].tolist()

inputs = p.get_features_from_texts(texts)
inputs = p.add_input_context(
    inputs, ["abstract"] * len(texts)
)

kwargs = {
    name: tf.convert_to_tensor(inputs[name], dtype=tf.int64)
    for name in ["input_ids", "input_mask", "segment_ids"]
}

kwargs["mlm_positions"] = tf.zeros(
    (len(texts), p.max_preds_per_seq),
    dtype=tf.int64
)

original_tf32 = (
    tf.config.experimental.tensor_float_32_execution_enabled()
)

results = {}

try:
    for enabled in [False, True]:
        tf.config.experimental.enable_tensor_float_32_execution(enabled)

        # Warm up
        p.model(**kwargs)["cls_token"].numpy()

        times = []
        for _ in range(5):
            start = time.perf_counter()
            output = p.model(**kwargs)["cls_token"].numpy()
            times.append(time.perf_counter() - start)

        results[enabled] = output.copy()

        print(
            f"TF32={enabled}: "
            f"{np.median(times):.3f} seconds/batch | "
            f"{256/np.median(times):.1f} abstracts/sec"
        )
finally:
    tf.config.experimental.enable_tensor_float_32_execution(
        original_tf32
    )

a = results[False].astype(np.float32)
b = results[True].astype(np.float32)

cosine = np.sum(a*b, axis=1) / (
    np.linalg.norm(a, axis=1) *
    np.linalg.norm(b, axis=1)
)

print("Minimum cosine similarity:", cosine.min())
print("Mean cosine similarity:", cosine.mean())

TF32=False: 7.314 seconds/batch | 35.0 abstracts/sec
TF32=True: 2.912 seconds/batch | 87.9 abstracts/sec
Minimum cosine similarity: 0.9999984
Mean cosine similarity: 0.99999976


In [28]:

import time
import gc
import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision
from api.bertfns import BertPredictor

# Keep other optimizations identical
tf.config.experimental.enable_tensor_float_32_execution(True)
tf.config.optimizer.set_jit(True)

# Prepare identical inputs
texts = df["abstract"].iloc[:256].tolist()

p = be.bert_predictor
inputs = p.get_features_from_texts(texts)
inputs = p.add_input_context(inputs, ["abstract"] * len(texts))

kwargs = {
    k: tf.convert_to_tensor(inputs[k], dtype=tf.int64)
    for k in ["input_ids", "input_mask", "segment_ids"]
}
kwargs["mlm_positions"] = tf.zeros((256, 45), dtype=tf.int64)

results = {}
original_policy = mixed_precision.global_policy().name

try:
    for policy in ["float32", "mixed_float16"]:

        mixed_precision.set_global_policy(policy)

        # Reload model after changing policy
        predictor = BertPredictor(
            model_name=be.MODEL_DIR,
            text_tokenizer=be.tokenizer,
            max_seq_length=be.MAX_SEQ_LENGTH,
            max_preds_per_seq=be.MAX_PREDS_PER_SEQUENCE,
            has_context=True
        )

        # Warm-up
        for _ in range(3):
            predictor.model(**kwargs)["cls_token"].numpy()

        # Benchmark
        times = []
        for _ in range(10):
            start = time.perf_counter()
            output = predictor.model(**kwargs)["cls_token"].numpy()
            times.append(time.perf_counter() - start)

        median_time = np.median(times)
        speed = 256 / median_time

        results[policy] = {
            "speed": speed,
            "embeddings": output.copy(),
            "dtype": output.dtype
        }

        print(
            f"{policy}: {speed:.1f} abstracts/sec | "
            f"{median_time:.3f}s/batch | "
            f"Output dtype: {output.dtype}"
        )

        del predictor
        gc.collect()

finally:
    mixed_precision.set_global_policy(original_policy)

# Compare performance
speedup = (
    results["mixed_float16"]["speed"]
    / results["float32"]["speed"]
)

print(f"\nMixed precision speedup: {speedup:.2f}x")

# Check numerical differences
a = results["float32"]["embeddings"].astype(np.float32)
b = results["mixed_float16"]["embeddings"].astype(np.float32)

cosine = np.sum(a*b, axis=1) / (
    np.linalg.norm(a, axis=1) *
    np.linalg.norm(b, axis=1)
)

print(f"Mean cosine similarity: {cosine.mean():.8f}")
print(f"Minimum cosine similarity: {cosine.min():.8f}")
print(f"Maximum absolute difference: {np.max(np.abs(a-b)):.8f}")


float32: 181.3 abstracts/sec | 1.412s/batch | Output dtype: float32
mixed_float16: 181.5 abstracts/sec | 1.411s/batch | Output dtype: float32

Mixed precision speedup: 1.00x
Mean cosine similarity: 1.00000000
Minimum cosine similarity: 0.99999994
Maximum absolute difference: 0.00000000


In [29]:
import time
import numpy as np

texts = df["abstract"].iloc[:256].tolist()

# Warm-up
be.get_embd_of_whole_abstract(texts)

times = []

for _ in range(5):
    start = time.perf_counter()

    be.get_embd_of_whole_abstract(texts)

    times.append(time.perf_counter() - start)

print(f"Full pipeline: {256 / np.median(times):.1f} abstracts/sec")

There are 256 patents being fed into BERT
It took 0.03539 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.00712 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.00712 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.00712 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.00712 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.00712 seconds to get the embeddings of the input.
Full pipeline: 140.3 abstracts/sec


In [37]:
import time
import numpy as np
import tensorflow as tf

# Enable optimizations
tf.config.experimental.enable_tensor_float_32_execution(True)
tf.config.optimizer.set_jit(True)

# Same 4,096 abstracts for every test
texts = df["abstract"].iloc[:4096].tolist()

results = []

for bs in [32, 64, 128, 256, 512, 1024]:
    try:
        # Warm-up
        be.get_embd_of_whole_abstract(texts[:bs])

        start = time.perf_counter()

        for i in range(0, len(texts), bs):
            be.get_embd_of_whole_abstract(texts[i:i+bs])

        elapsed = time.perf_counter() - start
        speed = len(texts) / elapsed

        results.append({
            "batch_size": bs,
            "abstracts_per_sec": round(speed, 2),
            "seconds_per_batch": round(elapsed / np.ceil(len(texts) / bs), 3)
        })

        print(f"Batch {bs}: {speed:.1f} abstracts/sec\n")

    except tf.errors.ResourceExhaustedError:
        print(f"Batch {bs}: GPU out of memory")
        tf.keras.backend.clear_session()
        break

# Results ranked by speed
benchmark = pd.DataFrame(results).sort_values(
    "abstracts_per_sec", ascending=False
)

display(benchmark)
print(f"\nOptimal batch size: {benchmark.iloc[0]['batch_size']:.0f}")

There are 32 patents being fed into BERT
It took 0.00828 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00749 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00752 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00749 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00747 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00727 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00732 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00737 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00744 seconds to get the embeddings of the input.
There are 32 patents being fed into BERT
It took 0.00740 seconds to get the embeddings of t

,batch_size,abstracts_per_sec,seconds_per_batch
4,512,140.43,3.646
3,256,140.28,1.825
2,128,139.35,0.919
0,32,134.40,0.238
1,64,74.81,0.856



Optimal batch size: 512


In [38]:
print(be.bert_predictor.model.structured_input_signature)

((), {'input_ids': TensorSpec(shape=(None, 512), dtype=tf.int64, name='input_ids'), 'input_mask': TensorSpec(shape=(None, 512), dtype=tf.int64, name='input_mask'), 'mlm_positions': TensorSpec(shape=(None, 45), dtype=tf.int64, name='mlm_positions'), 'segment_ids': TensorSpec(shape=(None, 512), dtype=tf.int64, name='segment_ids')})


In [39]:
for name, spec in be.bert_predictor.model.structured_outputs.items():
    print(name, spec.shape, spec.dtype)

mlm_logits (None, 39859) <dtype: 'float32'>
next_sentence_logits (None, 2) <dtype: 'float32'>
cls_token (None, 1024) <dtype: 'float32'>
encoder_layer (None, 512, 1024) <dtype: 'float32'>


In [40]:
sample = df["abstract"].sample(
    n=min(10000, len(df)),
    random_state=42
)

lengths = np.array([
    len(be.tokenizer.tokenize(text)) + 3
    for text in sample
])

print("Median:", np.median(lengths))
print("90th percentile:", np.percentile(lengths, 90))
print("95th percentile:", np.percentile(lengths, 95))
print("99th percentile:", np.percentile(lengths, 99))
print("Share <= 128:", np.mean(lengths <= 128))
print("Share <= 256:", np.mean(lengths <= 256))
print("Share <= 384:", np.mean(lengths <= 384))

Median: 133.0
90th percentile: 198.0
95th percentile: 231.0
99th percentile: 299.0
Share <= 128: 0.4705
Share <= 256: 0.9711
Share <= 384: 0.997


In [36]:
import time
import numpy as np

texts = df["abstract"].iloc[:2048].tolist()

for jit in [False, True]:
    tf.config.optimizer.set_jit(jit)

    # Warm-up
    be.get_embd_of_whole_abstract(texts[:256])

    start = time.perf_counter()

    for i in range(0, len(texts), 256):
        be.get_embd_of_whole_abstract(texts[i:i+256])

    elapsed = time.perf_counter() - start

    print(f"XLA={jit}: {len(texts)/elapsed:.1f} abstracts/sec")

There are 256 patents being fed into BERT
It took 0.01883 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01296 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01282 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01274 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01295 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01298 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01301 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01301 seconds to get the embeddings of the input.
There are 256 patents being fed into BERT
It took 0.01307 seconds to get the embeddings of the input.
XLA=False: 77.2 abstracts/sec
There are 256 patents being fed into BERT
It took 0.

# Combine with previously processed patents

In [ ]:
#@title Load all patents
previous_patents = load(r"/content/drive/MyDrive/PhD Data/01 CLS Embeddings/CheckedPatents_CLSonly_2024.11.11_00:06.pkl")
current_patents = load(r"/content/drive/MyDrive/PhD Data/01 CLS Embeddings/01 CLS embeddings rest - 20250226_163344.pkl")

assert(len(previous_patents) == 2077783)
assert(len(current_patents) == 6997638)


In [ ]:
#@title Combine, check and save

# Squeeze
for key, value in previous_patents.items():
    previous_patents[key] = value.squeeze()


# Combine
current_patents.update(previous_patents)
previous_patents = None



In [ ]:
# Get a list of all variables
all_vars = list(globals().keys())
print(all_vars)


['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'drive', '_i2', '_i3', 'time', 'pd', 'Dataset', 'DataLoader', 'os', '_exit_code', 'pickle', 'datetime', 'np', 'PatentDataset', 'collate_fn', 'save_embedding_dict', 'load', '_i4', 'previous_patents', 'current_patents', '_i5', 'key', 'value', '_i6', '_6', '_i7', '_7', '_i8', 'gc', 'sys', 'get_memory_variables', 'name', 'var', '_i9']


In [ ]:


#@title Check

#assert(len(current_patents) == 2777783 + 6997638)
print(len(current_patents))

# Check types
size_types  = set()
for key, value in current_patents.items():
    size_types.add(value.shape)

print(size_types)
assert(len(size_types) == 1)

9075421
{(1024,)}


In [ ]:
import dill as pickle
import os

# Define batch size
num_batches = 16  # Adjust based on testing
batch_size = len(current_patents) // num_batches

# Define save path (Change to your Google Drive path if using Colab)
save_dir = "/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings"
os.makedirs(save_dir, exist_ok=True)

# Save in batches
for i in range(num_batches):
    start = i * batch_size
    end = (i + 1) * batch_size if i != num_batches - 1 else len(current_patents)

    batch_dict = dict(list(current_patents.items())[start:end])

    with open(f"{save_dir}/batch_{i}.pkl", "wb") as f:
        pickle.dump(batch_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved batch {i+1}/{num_batches} - Size: {len(batch_dict)} items")


Saved batch 1/16 - Size: 567213 items
Saved batch 2/16 - Size: 567213 items
Saved batch 3/16 - Size: 567213 items
Saved batch 4/16 - Size: 567213 items
Saved batch 5/16 - Size: 567213 items
Saved batch 6/16 - Size: 567213 items
Saved batch 7/16 - Size: 567213 items
Saved batch 8/16 - Size: 567213 items
Saved batch 9/16 - Size: 567213 items
Saved batch 10/16 - Size: 567213 items
Saved batch 11/16 - Size: 567213 items
Saved batch 12/16 - Size: 567213 items
Saved batch 13/16 - Size: 567213 items
Saved batch 14/16 - Size: 567213 items
Saved batch 15/16 - Size: 567213 items
Saved batch 16/16 - Size: 567226 items


# Combine, adjust precision and save to save memory

In [7]:
import os
import glob
import pickle
import numpy as np

# Define the folder containing your batched embeddings.
embedding_batches_folder = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings'
# Define the output file path for the combined embeddings.
output_file = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings/All embeddings - float16/all_embeddings_float16.pkl'

combined_embeddings = {}

# Get list of all batch files (assumed to have a .pkl extension)
batch_files = glob.glob(os.path.join(embedding_batches_folder, '*.pkl'))

print(f"Found {len(batch_files)} batch files. Processing...")

for batch_file in batch_files:
    print(f"Loading {batch_file}...")
    with open(batch_file, 'rb') as f:
        batch_dict = pickle.load(f)
    # Ensure keys are strings and values are NumPy arrays.
    for k, v in batch_dict.items():
        combined_embeddings[str(k)] = np.array(v)

print("All batch files loaded. Converting embeddings to float16...")

# Convert every embedding to float16.
for k in combined_embeddings:
    combined_embeddings[k] = combined_embeddings[k].astype(np.float16)

print("Conversion complete. Saving the combined embeddings to file...")

# Save the combined dictionary as a single pickle file.
with open(output_file, 'wb') as f:
    pickle.dump(combined_embeddings, f)

print(f"Combined embeddings saved to {output_file}.")

# Calculate the total size of combined_embeddings in bytes and print it in MB.
total_size_bytes = sum(emb.nbytes for emb in combined_embeddings.values())
print(f"Total size of combined_embeddings: {total_size_bytes / (1024**2):.2f} MB")



Found 16 batch files. Processing...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_0.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_1.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_2.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_3.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_4.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_5.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_6.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_7.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_8.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_9.pkl...
Loading /content/drive/MyDrive/PhD Data/01 C